# React — JSX

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> You have been writing JSX since LESSON 4 without being told its rules. This topic is
> those rules.

## LESSON 7 — What JSX actually is

React's own definition:

> JSX is a syntax extension for JavaScript that lets you write HTML-like markup inside a
> JavaScript file.

Three things follow from that, and all three matter.

**It is not HTML.** It looks like it and it is stricter than it. The rules are in the next
three lessons.

**It is not a string.** `<p>Hello</p>` has no quotes around it. It is syntax, like `+` or
`=>`, and it is part of the file's grammar.

**Browsers do not understand it.** Your build tool — Vite, through its React plugin —
transforms JSX into ordinary JavaScript *before* the browser ever sees the file. What it
produces is calls into React's **JSX runtime**:

```jsx
// you write                    the build tool produces
<p>Hello</p>              →     jsx("p", { children: "Hello" })
<Greeting />              →     jsx(Greeting, {})
```

The exact function name varies — `jsx`, `jsxs` for several children, `jsxDEV` in
development builds, all from `react/jsx-runtime`. The shape never changes.

You saw this in LESSON 4 without knowing it: a capital `<Greeting />` compiles to the
**identifier** `Greeting`, your function; a lowercase `<greeting />` compiles to the
**string** `"greeting"`, an HTML tag name. That is the whole mechanism behind the naming
rule.

### What those calls return

Plain JavaScript objects. React calls them **elements**, and an element is a *description*
of what should appear — not a DOM node, and not anything on screen yet:

```js
{ type: "p", props: { children: "Hello" }, key: null, … }
```

That is the idea worth keeping. Writing JSX builds data. Rendering happens later, when
React takes that data and decides what to do with the real page. It is also why JSX can be
put in a variable, returned from a function, or stored in an array — it is just a value.

### Key Notes

- JSX is a **syntax extension for JavaScript**: HTML-like markup inside a `.jsx` file.
  Not HTML, not a string.
- The build tool transforms it into calls to React's JSX runtime before the browser sees it.
- Those calls return plain objects — **elements** — that describe UI. Nothing is rendered
  yet.
- Capitalised name compiles to an identifier (your component); lowercase compiles to a
  string (an HTML tag).

### Example

**Runnable — plain JS.** **Conceptual model — not React's actual implementation.**

The point of this cell is one claim: *an element is data you can inspect*. The real thing
carries extra fields (`key`, an internal `$$typeof` marker and some debug bookkeeping), so
do not read `l7h` as React's source. Read it as evidence that a UI description is an
ordinary object.

In [ ]:
// Conceptual model — not React's actual implementation.
function l7h(type, props = {}, ...children) {
  return { type, props: { ...props, children } };
}

const l7tree = l7h(
  "section",
  {},
  l7h("h2", {}, "Today"),
  l7h("p", {}, "Fresh bread"),
);

console.log(JSON.stringify(l7tree, null, 2));

// Nothing has been rendered. It is a value, like any other.
console.log(typeof l7tree, Array.isArray(l7tree.props.children));

### Exercise

Using `l7h` from the example cell, do both parts in the cell below.

**Part 1.** Build the element tree that this JSX would produce:

```jsx
<section>
  <h2>Today</h2>
  <p>Fresh bread</p>
  <p>Coffee beans</p>
</section>
```

Call it `l7today` and log it.

**Part 2.** Write `l7countElements(node)`, which returns how many **elements** a tree
contains. Plain strings like `"Coffee beans"` are not elements and must not be counted.

For the tree above the answer is **4**. Log the result to prove it.

Hint: in this model `node.props.children` is always an array, and each child in it is
either another element object or a plain string.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

No new code — reading the other direction.

1. Write out, by hand, the calls this JSX compiles to. Use `jsx(type, props)` notation, and
   nest them the way the build tool would:

   ```jsx
   <Card>
     <h2>Revenue</h2>
   </Card>
   ```

2. In your answer, `Card` appears without quotes and `h2` appears with them. Explain in one
   sentence what would go wrong if the build tool quoted both.

3. `const box = <p>Hi</p>;` is legal JavaScript in a `.jsx` file. Given what JSX compiles
   to, what is `box` — and has anything appeared on screen at that moment?

In [ ]:
// Your code here

## LESSON 8 — Braces hold expressions, not statements

So far your JSX has contained fixed text. Curly braces are how JavaScript gets in:

**The React API:**

```jsx
<p>Hello {user.name}, you have {user.visits} visits</p>
```

Everything between `{` and `}` is evaluated as JavaScript, and the result is placed there.

### The one rule

Braces take an **expression** — something that *produces a value*. They do not take a
**statement** — something that *does* something.

| expression — produces a value | statement — does something |
|---|---|
| `user.name` | `if (a) { … }` |
| `2 + 2` | `for (…) { … }` |
| `formatPrice(12.5)` | `const x = 1` |
| `items.length` | `return x` |
| `a > 0 ? "yes" : "no"` | `while (…) { … }` |

The test that never fails: **could it go on the right-hand side of `const result = …`?**
If yes, it can go inside braces.

`const result = if (loggedIn) { "Hi" }` is a syntax error in plain JavaScript — which is
exactly why this does not work either:

```jsx
// Not allowed. `if` is a statement.
<p>{ if (loggedIn) { "Hi" } }</p>
```

Ternaries and `.map()` are expressions, so both are allowed. Topic 7 is about using them
well; for now only the expression-vs-statement line matters.

### Where the logic goes instead

If you need a statement, run it **above** the `return`, and put the resulting value in
braces:

```jsx
function Greeting() {
  const hour = new Date().getHours();

  let greeting = "Good evening";
  if (hour < 12) greeting = "Good morning";

  return <p>{greeting}</p>;
}
```

This is the normal shape of a React component, and it is worth getting used to early:
plain JavaScript first, JSX at the end.

### Key Notes

- `{ }` evaluates a JavaScript **expression** and places the result.
- Expression = produces a value. Statement = does something. Only expressions fit.
- The test: could it follow `const result = …`?
- Statements go above the `return`; the value they produce goes in the braces.

### Example

**Runnable — plain JS.** Every line below produces a value, so every one of them would be
legal inside braces in JSX. Nothing here is React-specific — that is the point.

In [ ]:
const l8user = { name: "Ada", visits: 3 };

console.log(l8user.name);
console.log(l8user.visits * 2);
console.log(`${l8user.name} has ${l8user.visits} visits`);
console.log(l8user.visits > 0 ? "returning" : "new");
console.log([1, 2, 3].map((n) => n * 2).join(", "));

// Each of these survives the test — it can sit after `const result =`.
const l8result = l8user.visits > 0 ? "returning" : "new";
console.log(l8result);

### Exercise

Each snippet below uses a **statement**, so as written it cannot go inside JSX braces.
Rewrite each one as a single **expression**, assign it to the given variable, and log it.

**1 — `l8status`.** Same result as:

```js
let status;
if (score >= 60) {
  status = "pass";
} else {
  status = "fail";
}
```

Use `const l8score = 72;`.

**2 — `l8list`.** Same result as:

```js
let list = "";
for (const item of items) {
  list += item + " / ";
}
```

Use `const l8items = ["bread", "milk", "eggs"];` and produce `"bread / milk / eggs"` —
note there is no trailing separator, unlike the loop above.

**3 — `l8total`.** The sum of all prices in
`const l8prices = [4.5, 2.25, 9];`, formatted as `"15.75 EUR"`.

Then answer in a comment: which of the three original versions could have been written
directly inside JSX braces?

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each fragment, decide **expression** or **statement**. For every statement, write one
expression that produces the same value.

1. `items.filter((i) => i.done).length`
2. `let total = 0`
3. `user ? user.email : "no email"`
4. `if (isAdmin) { showPanel() }`
5. `prices.reduce((a, b) => a + b, 0)`
6. `for (const t of tags) { out.push(t) }`

Then answer this, in one sentence. Fragment 4 and fragment 3 both choose between two
outcomes. Only one of them can go inside JSX braces. What is the actual difference — not
"one is an if" — in terms of what each fragment *gives back*?

In [ ]:
// Your code here

## LESSON 9 — Attributes

JSX attributes look like HTML attributes, with three differences that cause almost all the
confusion.

### 1. Two names are different

`class` and `for` are reserved words in JavaScript, so JSX uses the DOM property names
instead:

```jsx
<div className="card">
  <label htmlFor="email">Email</label>
</div>
```

`className` is the React spelling; `class` is not. Write `class` anyway and React warns:

```text
Invalid DOM property `class`. Did you mean `className`?
```

Read that warning carefully, because the class **is** still applied. React passes attributes
it does not recognise straight through to the DOM, so the element really does end up with
`class="card"` and your CSS really does work. The only symptom is the console warning —
which is exactly why this mistake survives in real codebases for months. Use `className`.

### 2. Names are camelCase

`tabIndex`, `maxLength`, `readOnly`, `strokeWidth`. Event attributes follow the same rule:
`onClick`, not `onclick`. Topic 8 is about actually using those; here only the spelling
matters.

### 3. Values are strings or braces

```jsx
<img src="/logo.png" width={120} alt="" />
```

A quoted string is a literal. Braces mean "evaluate this JavaScript", under exactly the
rule from LESSON 8. `width="120"` and `width={120}` both work here; `width={size}` is only
possible with braces.

### `style` takes an object

Not a string:

```jsx
<p style={{ fontSize: 16, borderRadius: 4 }}>Hi</p>
```

The outer braces are JSX's "here comes JavaScript". The inner braces are an ordinary object
literal. Keys are camelCase, and **a number gets `px` appended automatically** — unless the
property is unitless, like `opacity`, `zIndex`, `lineHeight` or `flex`.

React's own advice is worth repeating: use `style` for values you cannot know ahead of
time, and `className` for everything else. Topic 4 covers styling properly.

### Spreading a set of attributes

An object of attributes can be applied in one go:

```jsx
const inputAttrs = { type: "email", required: true, maxLength: 80 };

<input {...inputAttrs} placeholder="you@example.com" />
```

Anything written **after** the spread wins, because these become object properties and the
last one assigned takes effect.

### Key Notes

- `className` and `htmlFor` replace `class` and `for` — both are reserved words.
- Attribute names are camelCase; values are `"literal strings"` or `{expressions}`.
- `style` takes an **object** with camelCase keys; numbers get `px` unless the property is
  unitless.
- `{...obj}` applies a set of attributes at once; later attributes override earlier ones.

### Example

**Runnable — plain JS.** The camelCase-to-CSS translation is mechanical, and doing it by
hand once makes React's style object stop feeling arbitrary.

In [ ]:
const l9unitless = new Set(["opacity", "zIndex", "lineHeight", "flex", "fontWeight"]);

function l9toCss(style) {
  return Object.entries(style)
    .map(([key, value]) => {
      const prop = key.replace(/[A-Z]/g, (letter) => "-" + letter.toLowerCase());
      const out = typeof value === "number" && !l9unitless.has(key) ? `${value}px` : value;
      return `${prop}: ${out}`;
    })
    .join("; ");
}

console.log(l9toCss({ fontSize: 16, borderRadius: 4, backgroundColor: "papayawhip" }));
console.log(l9toCss({ opacity: 0.5, zIndex: 10 }));

### Exercise

Two parts, both in the cell below.

**Part 1 — merging attributes.** You have a set of defaults and a set of overrides:

```js
const l9defaults = { type: "text", required: false, maxLength: 50 };
const l9overrides = { required: true, maxLength: 80 };
```

Produce `l9attrs`, a single object with the overrides applied on top, using spread. Log it.
Then log `l9defaults` and confirm it was **not** changed.

**Part 2 — one dynamic style.** Write `l9barStyle(percent)`, which returns a style object
for a progress bar:

- `width` is a percentage **string**, like `"45%"` — passing the plain number `45` would
  become `45px`, which is not what a progress bar wants;
- `backgroundColor` is `"seagreen"` when `percent` is 70 or more, otherwise `"goldenrod"`;
- `opacity` is `1` when `percent` is above 0, otherwise `0.4`.

Log `l9barStyle(45)` and `l9barStyle(90)`, then pass both through `l9toCss` to see what the
browser would receive.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Five attributes. For each: is it correct JSX? If not, what is wrong and what does React do
about it?

1. `<div class="card">`
2. `<label for="email">`
3. `<input maxlength={10} />`
4. `<div style="color: red">`
5. `<img width="100" alt="" />`

Then two questions:

- `style={{ fontSize: 16 }}` has two sets of braces. Say exactly what each one is doing.
- Why can React get away with `opacity: 1` meaning `1`, but `fontSize: 16` meaning `16px`?
  What would it have to do if it did not keep a list?

In [ ]:
// Your code here

## LESSON 10 — Fragments, closing tags, comments, and what actually renders

Four small rules that finish JSX off.

### Fragments

LESSON 4 promised a tidier wrapper than `<div>`. This is it:

```jsx
return (
  <>
    <h1>Dashboard</h1>
    <p>Everything at a glance</p>
  </>
);
```

`<>…</>` is a **fragment**. It groups elements so a component can return several, and it
leaves **nothing** in the browser's HTML — no extra `<div>` in the page.

That matters more than it sounds. A stray `<div>` between a flex or grid container and its
children changes the layout, because the children are now grandchildren. Fragments avoid
inventing elements that only exist to satisfy the "one root" rule.

The long form `<Fragment>…</Fragment>` also exists, and you need it in one case: when the
fragment requires a `key`. Topic 7 gets there.

### Close every tag

JSX has no optional closing tags. `<img />`, `<br />`, `<input />`, `<hr />` — all
self-closed. `<li>oranges` must be `<li>oranges</li>`.

### Comments

Inside JSX you are in markup, so braces get you back to JavaScript first:

```jsx
<div>
  {/* this is a JSX comment */}
  <p>Visible</p>
</div>
```

Write `// like this` between tags and it renders on the page as the text `// like this`.

### What each value renders

Braces accept any expression, so it is worth knowing what React does with the result:

| value | renders |
|---|---|
| `"text"` | the text |
| `42` | `42` |
| `0` | **`0`** — a number is a number |
| `null` | nothing |
| `undefined` | nothing |
| `true` / `false` | nothing |
| `["a", "b"]` | each item, in order |
| `{ name: "Ada" }` | **throws**: *Objects are not valid as a React child* |

React's words for the empty cases: it "considers `false` as a hole in the JSX tree, just
like `null` or `undefined`, and doesn't render anything in its place".

Remember the `0` row. It is the one that surprises people, and topic 7 shows the bug it
causes.

### Key Notes

- `<>…</>` groups elements and adds nothing to the page; `<div>` adds an element.
- Every tag closes, including `<img />` and `<input />`.
- Comments inside JSX are `{/* … */}`.
- `null`, `undefined` and booleans render nothing. Numbers and strings render — **including
  `0`**. Arrays render each item. A plain object throws.

### Example

**Runnable — plain JS.** The table above, written as a function. Encoding a rule as code is
the fastest way to stop half-remembering it.

In [ ]:
function l10renders(value) {
  if (value === null || value === undefined) return "";
  if (typeof value === "boolean") return "";
  if (Array.isArray(value)) return value.map(l10renders).join("");
  if (typeof value === "object") throw new TypeError("Objects are not valid as a React child");
  return String(value);
}

console.log(JSON.stringify(l10renders("text")));
console.log(JSON.stringify(l10renders(42)));
console.log(JSON.stringify(l10renders(0)));     // "0" — not empty
console.log(JSON.stringify(l10renders(null)));
console.log(JSON.stringify(l10renders(false)));
console.log(JSON.stringify(l10renders(["a", "b"])));

### Exercise

Write `l10describe(value)`, which returns a short **description** of what React would do —
not the rendered text itself. It must return exactly one of these strings:

| case | returns |
|---|---|
| `null`, `undefined`, `true`, `false` | `"nothing"` |
| a string or a number | `"text: <the value>"` — so `0` gives `"text: 0"` |
| an array | `"list of N"`, where N is the number of items |
| a plain object | `"error"` |

Then run it over this list and log each result on its own line:

```js
const l10values = [0, "", "Ada", null, undefined, false, true, [1, 2, 3], { name: "Ada" }];
```

Two things to get right: an empty string is still text, and `Array.isArray` has to be
checked **before** `typeof value === "object"` — arrays are objects too.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Four questions. Answers in comments, and code only where it helps.

1. `<div>` and `<>` both satisfy the one-root rule. Describe a concrete situation where
   swapping `<>` for `<div>` visibly breaks a page.
2. A colleague writes `// TODO: add avatar` on its own line between two JSX tags. What does
   the user see?
3. `const l10empty = [];` and the JSX is `<div>{l10empty}</div>`. What ends up in the page —
   and how is that different from `<div>{null}</div>`?
4. `<p>{user}</p>` where `user = { name: "Ada" }` fails. What did the author almost
   certainly mean, and why does React refuse rather than guessing?

In [ ]:
// Your code here